In [ ]:
import cv2
import numpy as np
from datetime import datetime

# ============================================
# CONFIGURAÇÕES DOS FILTROS (altere aqui facilmente)
# ============================================

# Filtro considerado "melhor" (exemplo: mediana com kernel 5)
melhor_filtro = 'median'  # Opções: 'median', 'gaussian', 'bilateral', 'mean'
melhor_kernel = 5          # Tamanho do kernel (ímpar para mediana)

# Filtro considerado "pior" (exemplo: média com kernel 11)
pior_filtro = 'mean'       # Opções: 'median', 'gaussian', 'bilateral', 'mean'
pior_kernel = 11           # Tamanho do kernel

# ============================================
# Funções de filtragem
# ============================================

def apply_filter(img, filter_type, kernel_size):
    """Aplica o filtro especificado à imagem."""
    if filter_type == 'mean':
        kernel = np.ones((kernel_size, kernel_size), np.float32) / (kernel_size * kernel_size)
        return cv2.filter2D(img, -1, kernel)
    elif filter_type == 'gaussian':
        return cv2.GaussianBlur(img, (kernel_size, kernel_size), 0)
    elif filter_type == 'median':
        return cv2.medianBlur(img, kernel_size)
    elif filter_type == 'bilateral':
        # Para bilateral, o kernel_size é usado como diâmetro e os parâmetros de sigma
        return cv2.bilateralFilter(img, kernel_size, kernel_size*2, kernel_size//2)
    else:
        return img

# ============================================
# Inicialização da webcam
# ============================================
cap = cv2.VideoCapture(0)  # 0 para a webcam padrão
if not cap.isOpened():
    print("Erro ao abrir a webcam")
    exit()

print("Pressione 's' para salvar as imagens (original + filtradas)")
print("Pressione 'q' para sair")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Falha ao capturar frame")
        break

    # Aplica os dois filtros
    img_melhor = apply_filter(frame, melhor_filtro, melhor_kernel)
    img_pior = apply_filter(frame, pior_filtro, pior_kernel)

    # Exibe as imagens em janelas
    cv2.imshow('Original', frame)
    cv2.imshow('Melhor Filtro', img_melhor)
    cv2.imshow('Pior Filtro', img_pior)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('s'):
        # Salva as imagens com timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        cv2.imwrite(f"original_{timestamp}.jpg", frame)
        cv2.imwrite(f"melhor_{melhor_filtro}_{melhor_kernel}_{timestamp}.jpg", img_melhor)
        cv2.imwrite(f"pior_{pior_filtro}_{pior_kernel}_{timestamp}.jpg", img_pior)
        print(f"Imagens salvas em {timestamp}")

    elif key == ord('q'):
        break

# Libera recursos
cap.release()
cv2.destroyAllWindows()